In [ ]:
#pour eeger meilleur k
import numpy as np
from sklearn.cluster import KMeans
import matplotlib.pyplot as plt

def find_best_k_using_elbow(data, k_range=range(1, 11), plot=True):
    inertias = []

    # Étape 1 : Calcul des inerties pour chaque k
    for k in k_range:
        kmeans = KMeans(n_clusters=k, random_state=42)
        kmeans.fit(data)
        inertias.append(kmeans.inertia_)

    # Étape 2 : Trouver le "coude" en utilisant la méthode de la plus grande distance
    k_vals = list(k_range)
    p1 = np.array([k_vals[0], inertias[0]])
    p2 = np.array([k_vals[-1], inertias[-1]])

    distances = []
    for i in range(len(k_vals)):
        p = np.array([k_vals[i], inertias[i]])
        distance = np.linalg.norm(np.cross(p2-p1, p1-p)) / np.linalg.norm(p2-p1)
        distances.append(distance)

    best_k = k_vals[np.argmax(distances)]

    return best_k


In [1]:
import pandas as pd 
df_global_seg=pd.read_csv("./dataframe_seg.csv",low_memory=False)

In [2]:
segmentation_guide={
    "rentabilité":["rentabilité","nb_achat","nb_recharge","frequence_utilisation_mensuelle","moy_recharge",'duree_moyenne_option'],
    "engagement":['est_utilisateur_actif','frequence_utilisation_mensuelle','engagement_score','engagement_level'],
    "type_client":['type_client','frequence_utilisation_mensuelle','type_usage'],
    "type_interet":['type_usage','interet','derniere_offre_achetee'],
    "interet_international":['type_usage','interet_international','source_achat'],
    "interet_jeu":['type_client','nombre_essai_jeu','win_count'],
    "interet_promo":['source_achat', 'interet_promo', 'type_usage','reponse_promo',],
    "action":['nombre_jeu','nb_recharge', 'nb_achat','action_majoritaire']
}

In [3]:
segments_labels = {
    'rentabilité': {
        'rentabilité_score': ['non rentable', 'rentable']
    },
    'engagement': {
        'engagement_score': ['non engagé','peu engagé', 'très engagé']
    },
    "type_client":{
        'type_client': ['orienté USSD', 'orienté APLICATION', 'orienté BOUTIQUE']
    },
    "type_interet":{  
        'interet':['data','voix']},
    "interet_international":{
        'interet_international':['non international', 'international']
    },
    "interet_jeu":{
        'nombre_essai_jeu':['non jeu','peu jeu','très jeu']
    },
    "interet_promo":{
        'interet_promo':['non promo','peu promo','Sensibles aux promos']
    },
    "action":{
        'action_majoritaire':['achat','recharge','roue chance']
    }
}

In [4]:
def assign_labels_by_feature(centroids, feature_index, ordered_labels):
    """
    Attribue des labels aux clusters selon la valeur d'une seule feature choisie.

    Args:
        centroids (np.ndarray): Matrice (n_clusters x n_features)
        feature_index (int): Index de la feature à utiliser pour le tri (ex: 0 pour 1re colonne)
        ordered_labels (list of str): Labels à assigner dans l’ordre croissant de la feature

    Returns:
        dict: Mapping des index de cluster vers labels
    """
    
    # Extraire la colonne (feature) choisie
    feature_values = centroids[:, feature_index]
    
    # Trier les indices de clusters selon cette feature
    sorted_indices = np.argsort(feature_values)
    
    # Assigner les labels dans l’ordre souhaité
    label_mapping = {idx: ordered_labels[i] for i, idx in enumerate(sorted_indices)}
    
    return label_mapping


In [5]:
from sklearn.preprocessing import LabelEncoder, MinMaxScaler
import numpy as np
import pandas as pd

def preprocess_dataframe(df, threshold=0.1):
    """
    Preprocesses the dataframe by:
    1. Label encoding categorical columns
    2. Normalizing numerical columns if necessary based on the threshold
    3. Standardizing numerical columns (excluding 'msisdn')
    
    Parameters:
    df (pd.DataFrame): Input dataframe to preprocess
    threshold (float): Threshold for determining whether to normalize numerical features (default 0.1)
    
    Returns:
    pd.DataFrame: Preprocessed dataframe
    """
    # Label encode categorical columns
    for col in df.columns:
        if df[col].dtype == object:
            le = LabelEncoder()
            df[col] = le.fit_transform(df[col])
    
    # Sélectionner les colonnes numériques (en excluant 'msisdn')
    numerical_cols = df.select_dtypes(include=['float64', 'int64']).columns.tolist()
    numerical_cols = [col for col in numerical_cols if col != 'msisdn']

    # Initialiser le scaler Min-Max
    scaler = MinMaxScaler()

    # Liste pour stocker les features qui nécessitent une normalisation
    features_to_normalize = []

    # Vérifier les différences entre les valeurs min et max pour chaque feature
    for col in numerical_cols:
        min_val = df[col].min()
        max_val = df[col].max()

        # Calculer la différence relative entre la valeur maximale et minimale
        diff = (max_val - min_val) / min_val if min_val != 0 else max_val

        # Si la différence est plus grande que le seuil, ajouter la feature à la liste
        if diff > threshold:
            features_to_normalize.append(col)

    # Appliquer la normalisation Min-Max si nécessaire
    if features_to_normalize:
        print(f"Normalisation des features suivantes: {features_to_normalize}")
        df[features_to_normalize] = scaler.fit_transform(df[features_to_normalize])
    else:
        print("Aucune normalisation nécessaire, les échelles des features sont déjà similaires.")

    return df

In [6]:
def normalize_and_pca(df, features):
    """
    Normalise les features et applique PCA
    
    Parameters:
    df (pd.DataFrame): Dataframe contenant les données
    features (list): Liste des features à normaliser
    
    Returns:
    pd.DataFrame: Dataframe normalisé
    """
    # Normalisation des features
    scaler = MinMaxScaler()
    df[features] = scaler.fit_transform(df[features])
    
    # Appliquer PCA
    pca = PCA()
    X_pca = pca.fit_transform(df[features])
    
    # Créer un nouveau dataframe avec les composantes principales
    df_pca = pd.DataFrame(X_pca, columns=[f'PC{i+1}' for i in range(X_pca.shape[1])])
    
    # Ajouter les colonnes non-numériques
    non_numeric_cols = [col for col in df.columns if col not in features]
    df_final = pd.concat([df[non_numeric_cols], df_pca], axis=1)
    
    # Afficher les informations
    print(f"Nombre total de features numériques: {len(features)}")
    print(f"Nombre de features sélectionnées: {X_pca.shape[1]}")
    print(f"Variance expliquée: {np.sum(pca.explained_variance_ratio_)*100:.2f}%")
    
    return df_final, pca

# Utilisation
features = ['rentabilite', 'nb_achat', 'nb_recharge', 'frequence_utilisation_mensuelle', 'moy_recharge', 'duree_moyenne_option']
df_normalized, pca = normalize_and_pca(df_global_seg, features)

# Visualiser les résultats du PCA
plt.figure(figsize=(10, 6))
plt.plot(range(1, len(pca.explained_variance_ratio_)+1), 
         np.cumsum(pca.explained_variance_ratio_), 
         marker='o')
plt.xlabel('Nombre de composantes principales')
plt.ylabel('Variance cumulée expliquée')
plt.title('Variance expliquée par composante')
plt.grid(True)
plt.show()

# Afficher la variance expliquée par composante
print("\nVariance expliquée par composante:")
for i, ratio in enumerate(pca.explained_variance_ratio_):
    print(f"PC{i+1}: {ratio*100:.2f}%")

ValueError: could not convert string to float: 'actif'

In [ ]:
from sklearn.cluster import KMeans
def segmenter_clients(df,crietere_segmentation,acquisition):
    if acquisition:
        df=df[df['type_client'] != 'maxit']
    else:
        df=df[df['type_client'] == 'maxit']
    features=segmentation_guide[crietere_segmentation]
    features = features + ['msisdn']
    df=df[features]
    df_seg=df.copy()
    df_seg=preprocess_dataframe(df_seg)
    feature_index = list(segments_labels[crietere_segmentation].keys())[0]
    ordered_labels=segments_labels[crietere_segmentation][feature_index]
    df_seg,_=select_features_pca(df_seg)
    best_k = find_best_k_using_elbow(df_seg)
    model = KMeans(init='k-means++',
               n_clusters=best_k,
               max_iter=500,
               random_state=22)
    segments = model.fit_predict(df_seg)
    df['segments']=segments
    centroids=model.cluster_centers_
    mapping =assign_labels_by_feature(centroids, feature_index, ordered_labels)
    df['segment_client'] = df['segments'].map(mapping)
    return df



In [ ]:
#evaluation de kmeans 
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, calinski_harabasz_score, davies_bouldin_score
from sklearn.metrics.pairwise import pairwise_distances_argmin_min
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

def evaluate_kmeans(X, n_clusters_range=(2, 10)):
    """
    Évalue le K-means sur différents nombres de clusters
    """
    # Initialiser les listes pour stocker les métriques
    metrics = {
        'n_clusters': [],
        'inertia': [],
        'silhouette': [],
        'calinski': [],
        'davies': [],
        'avg_distance': []
    }
    
    # Tester différents nombres de clusters
    for n in range(n_clusters_range[0], n_clusters_range[1] + 1):
        print(f"Évaluant avec {n} clusters...")
        
        # Créer et entraîner le modèle K-means
        kmeans = KMeans(n_clusters=n, random_state=42)
        kmeans.fit(X)
        labels = kmeans.labels_
        
        # Calculer les métriques
        metrics['n_clusters'].append(n)
        metrics['inertia'].append(kmeans.inertia_)  # Sum of squared distances
        
        if n > 1:  # Ces métriques nécessitent au moins 2 clusters
            metrics['silhouette'].append(silhouette_score(X, labels))
            metrics['calinski'].append(calinski_harabasz_score(X, labels))
            metrics['davies'].append(davies_bouldin_score(X, labels))
            
            # Calculer la distance moyenne aux centres
            avg_dist, _ = pairwise_distances_argmin_min(X, kmeans.cluster_centers_)
            metrics['avg_distance'].append(np.mean(avg_dist))
        else:
            metrics['silhouette'].append(0)
            metrics['calinski'].append(0)
            metrics['davies'].append(0)
            metrics['avg_distance'].append(0)
    
    # Créer un DataFrame des résultats
    results = pd.DataFrame(metrics)
    
    # Visualiser les métriques
    plt.figure(figsize=(15, 10))
    
    # 1. Méthode du coude (Inertia)
    plt.subplot(2, 2, 1)
    plt.plot(results['n_clusters'], results['inertia'], 'bo-')
    plt.title('Méthode du Coude')
    plt.xlabel('Nombre de clusters')
    plt.ylabel('Inertia')
    
    # 2. Silhouette Score
    plt.subplot(2, 2, 2)
    plt.plot(results['n_clusters'], results['silhouette'], 'go-')
    plt.title('Score de Silhouette')
    plt.xlabel('Nombre de clusters')
    plt.ylabel('Score')
    
    # 3. Calinski-Harabasz
    plt.subplot(2, 2, 3)
    plt.plot(results['n_clusters'], results['calinski'], 'ro-')
    plt.title('Score de Calinski-Harabasz')
    plt.xlabel('Nombre de clusters')
    plt.ylabel('Score')
    
    # 4. Davies-Bouldin
    plt.subplot(2, 2, 4)
    plt.plot(results['n_clusters'], results['davies'], 'mo-')
    plt.title('Score de Davies-Bouldin')
    plt.xlabel('Nombre de clusters')
    plt.ylabel('Score')
    
    plt.tight_layout()
    plt.show()
    
    return results

# Exemple d'utilisation
# evaluation_results = evaluate_kmeans(df_seg_type_client_filtered)

In [ ]:
df_action=segmenter_clients(df_global_seg,'rentabilité',acquisition=1)

NameError: name 'segmenter_clients' is not defined